In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
DATA_DIR = Path("../Data")
LOG_RETURNS_FILE = DATA_DIR / "crypto_and_crix_log_returns.csv"
EVENT_DATES_FILE = DATA_DIR / "event_dates.csv"
ASSET_TICKER = "ETH"
MARKET_INDEX_TICKER = "CRIX"

EVENT_MARKET_MODEL_RESULTS_FILE = DATA_DIR / "event_market_model_results_scholes_williams.csv"
PRE_EVENT_ABNORMAL_LOG_RETURNS_FILE = DATA_DIR / "pre_event_abnormal_log_returns_scholes_williams.csv"
ETH_EVENT_WINDOW_LOG_RETURNS_FILE = DATA_DIR / "eth_event_window_log_returns.csv"
CRIX_EVENT_WINDOW_LOG_RETURNS_FILE = DATA_DIR / "crix_event_window_log_returns.csv"
ETH_ABNORMAL_LOG_RETURNS_FILE = DATA_DIR / "eth_abnormal_log_returns_scholes_williams.csv"

ESTIMATION_WINDOW_START = 37
ESTIMATION_WINDOW_END = 7

EVENT_WINDOW_BEFORE = 3
EVENT_WINDOW_AFTER = 21

Z_SCORE = 1.96

In [ ]:
log_returns = pd.read_csv(
    LOG_RETURNS_FILE,
    index_col="Date",
    parse_dates=["Date"],
).sort_index()

log_returns.head()

In [ ]:
required_columns = [ASSET_TICKER, MARKET_INDEX_TICKER]

missing_columns = [
    column
    for column in required_columns
    if column not in log_returns.columns
]

if missing_columns:
    raise KeyError(
        f"Missing columns: {missing_columns}. "
        f"Available columns: {log_returns.columns.tolist()}"
    )

eth_crix_log_returns = (
    log_returns[required_columns]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sort_index()
)

eth_crix_log_returns.head()

In [ ]:
print("Columns:")
print(eth_crix_log_returns.columns.tolist())

print("\nDate range:")
print(f"Start: {eth_crix_log_returns.index.min().date()}")
print(f"End:   {eth_crix_log_returns.index.max().date()}")

print("\nShape:")
print(eth_crix_log_returns.shape)

print("\nMissing values:")
print(eth_crix_log_returns.isna().sum())

In [ ]:
event_dates_df = pd.read_csv(
    EVENT_DATES_FILE,
    parse_dates=["event_date"],
)

if "event_date" not in event_dates_df.columns:
    raise KeyError("event_dates.csv must contain an 'event_date' column.")

EVENT_DATES = pd.DatetimeIndex(event_dates_df["event_date"]).normalize()

print(f"Number of events: {len(EVENT_DATES)}")
print("\nEvent dates:")
EVENT_DATES

In [ ]:
event_date_check = pd.DataFrame(
    {
        "event_date": EVENT_DATES,
        "exists_in_data": EVENT_DATES.isin(eth_crix_log_returns.index),
    }
)

event_date_check

In [ ]:
def estimate_event_market_models_scholes_williams(
    return_data,
    event_dates,
    days_before_start=37,
    days_before_end=7,
    asset_column="ETH",
    market_column="CRIX",
):
    """
    For each event date, fit the Scholes-Williams market model:
    
    ETH_t = alpha + beta_avg*CRIX_t + epsilon_t
    
    Where beta_avg = (beta_lagged + beta_current + beta_leading) / 3
    
    The full model with all three market return terms:
    ETH_t = alpha + beta_1*CRIX_{t-1} + beta_2*CRIX_t + beta_3*CRIX_{t+1} + epsilon_t
    
    Then: beta_avg = (beta_1 + beta_2 + beta_3) / 3
    
    This adjusts for non-synchronous trading between assets.
    
    Returns
    -------
    event_market_model_results : pd.DataFrame
        Event-level regression estimates and diagnostics.
        
    pre_event_abnormal_log_returns : pd.DataFrame
        Pre-event abnormal returns, one column per event.
    """
    return_data = return_data.copy()

    return_data.index = pd.to_datetime(return_data.index)
    return_data = return_data.sort_index()

    event_dates = pd.DatetimeIndex(pd.to_datetime(event_dates))

    result_records = []
    pre_event_ar_dict = {}

    for event_date in event_dates:
        window_start = event_date - pd.Timedelta(days=days_before_start)
        window_end = event_date - pd.Timedelta(days=days_before_end)

        estimation_window = return_data.loc[
            window_start:window_end,
            [asset_column, market_column],
        ].copy()

        estimation_window = (
            estimation_window
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if len(estimation_window) < 10:
            print(
                f"Skipping {event_date.date()}: "
                "insufficient data in estimation window."
            )
            result_records.append(
                {
                    "event_date": event_date,
                    "alpha": np.nan,
                    "beta_avg": np.nan,
                    "beta_lagged": np.nan,
                    "beta_current": np.nan,
                    "beta_leading": np.nan,
                    "r_squared": np.nan,
                    "residual_std_error": np.nan,
                }
            )
            continue

        # Extract returns
        asset_returns = estimation_window[asset_column].values
        market_returns = estimation_window[market_column].values

        # Create lagged, current, and leading market returns
        market_current = market_returns.reshape(-1, 1)
        market_lagged = np.roll(market_returns, 1).reshape(-1, 1)
        market_leading = np.roll(market_returns, -1).reshape(-1, 1)

        # Set first lagged value and last leading value to NaN
        market_lagged[0] = np.nan
        market_leading[-1] = np.nan

        # Create design matrix with all three market return terms
        X = np.hstack([market_current, market_lagged, market_leading])
        X = sm.add_constant(X, has_constant='add')

        # Remove rows with NaN values
        valid_idx = ~np.isnan(X).any(axis=1)
        X = X[valid_idx]
        y = asset_returns[valid_idx].reshape(-1, 1)

        if len(y) < 5:
            print(
                f"Skipping {event_date.date()}: "
                "insufficient valid data after creating lags."
            )
            result_records.append(
                {
                    "event_date": event_date,
                    "alpha": np.nan,
                    "beta_avg": np.nan,
                    "beta_lagged": np.nan,
                    "beta_current": np.nan,
                    "beta_leading": np.nan,
                    "r_squared": np.nan,
                    "residual_std_error": np.nan,
                }
            )
            continue

        # Fit the model
        model = sm.OLS(y, X)
        results = model.fit()

        # Extract coefficients
        alpha = results.params[0]
        beta_current = results.params[1]
        beta_lagged = results.params[2]
        beta_leading = results.params[3]

        # Calculate average beta (Scholes-Williams)
        beta_avg = (beta_lagged + beta_current + beta_leading) / 3

        # Store results
        result_records.append(
            {
                "event_date": event_date,
                "alpha": alpha,
                "beta_avg": beta_avg,
                "beta_lagged": beta_lagged,
                "beta_current": beta_current,
                "beta_leading": beta_leading,
                "r_squared": results.rsquared,
                "residual_std_error": np.sqrt(results.mse_resid),
            }
        )

        # Calculate pre-event abnormal returns
        expected_returns = alpha + beta_avg * estimation_window[market_column].values
        abnormal_returns = estimation_window[asset_column].values - expected_returns
        pre_event_ar_dict[event_date] = abnormal_returns

    # Convert results to DataFrame
    event_market_model_results = pd.DataFrame(result_records)

    # Create pre-event abnormal returns DataFrame
    pre_event_abnormal_log_returns = pd.DataFrame(pre_event_ar_dict)

    return event_market_model_results, pre_event_abnormal_log_returns

In [ ]:
def build_event_log_return_windows(
    return_data,
    event_dates,
    limit_before=3,
    limit_after=21,
    asset_column="ETH",
    market_column="CRIX",
):
    """
    Extract fixed-length aligned ETH and CRIX return windows.

    Relative day:
        -3 = three observations before the event
         0 = event day
       +21 = twenty-one observations after the event.
    """
    return_data = return_data.copy()

    return_data.index = pd.to_datetime(return_data.index)
    return_data = return_data.sort_index()

    event_dates = pd.DatetimeIndex(pd.to_datetime(event_dates))

    relative_days = pd.Index(
        range(
            -int(limit_before),
            int(limit_after) + 1,
        ),
        name="relative_day",
    )

    expected_length = len(relative_days)

    asset_windows = {}
    market_windows = {}

    for event_date in event_dates:

        if event_date not in return_data.index:
            print(
                f"Skipping {event_date.date()}: "
                "event date not found."
            )
            continue

        event_position = return_data.index.get_loc(event_date)

        start_position = event_position - int(limit_before)
        end_position = event_position + int(limit_after) + 1

        if start_position < 0:
            print(
                f"Skipping {event_date.date()}: "
                "insufficient data before event."
            )
            continue

        if end_position > len(return_data):
            print(
                f"Skipping {event_date.date()}: "
                "insufficient data after event."
            )
            continue

        asset_window = return_data.iloc[
            start_position:end_position, return_data.columns.get_loc(asset_column)
        ].values

        market_window = return_data.iloc[
            start_position:end_position, return_data.columns.get_loc(market_column)
        ].values

        if len(asset_window) != expected_length:
            print(
                f"Skipping {event_date.date()}: "
                f"window length mismatch. Got {len(asset_window)}, expected {expected_length}."
            )
            continue

        asset_windows[event_date] = asset_window
        market_windows[event_date] = market_window

    asset_return_df = pd.DataFrame(
        asset_windows, index=relative_days
    )

    market_return_df = pd.DataFrame(
        market_windows, index=relative_days
    )

    return asset_return_df, market_return_df

In [ ]:
def compute_car_with_ci(
    asset_return_df,
    market_return_df,
    event_market_model_results,
    z_score=1.96,
):
    """
    Compute expected returns, abnormal returns, CAR, and
    approximate confidence bands.
    """
    asset_return_df = asset_return_df.copy()
    market_return_df = market_return_df.copy()
    event_market_model_results = event_market_model_results.copy()

    if "event_date" not in event_market_model_results.columns:
        if event_market_model_results.index.name == "event_date":
            event_market_model_results = event_market_model_results.reset_index()
        else:
            raise KeyError(
                "event_market_model_results must contain an 'event_date' column."
            )

    event_market_model_results["event_date"] = pd.to_datetime(
        event_market_model_results["event_date"]
    ).dt.normalize()

    asset_return_df.columns = pd.DatetimeIndex(
        pd.to_datetime(asset_return_df.columns)
    ).normalize()

    market_return_df.columns = pd.DatetimeIndex(
        pd.to_datetime(market_return_df.columns)
    ).normalize()

    regression_events = pd.DatetimeIndex(
        event_market_model_results["event_date"]
    )

    valid_events = (
        regression_events
        .intersection(asset_return_df.columns)
        .intersection(market_return_df.columns)
    )

    if len(valid_events) == 0:
        raise ValueError(
            "No common event dates were found across the inputs."
        )

    coefficient_df = event_market_model_results.set_index(
        "event_date"
    ).loc[valid_events, ["alpha", "beta_avg"]]

    expected_return_df = pd.DataFrame(
        index=asset_return_df.index,
        columns=valid_events,
        dtype="float64",
    )

    abnormal_return_df = pd.DataFrame(
        index=asset_return_df.index,
        columns=valid_events,
        dtype="float64",
    )

    for event_date in valid_events:
        alpha, beta = coefficient_df.loc[event_date, ["alpha", "beta_avg"]]

        expected_return_df[event_date] = (
            alpha + beta * market_return_df[event_date]
        )

        abnormal_return_df[event_date] = (
            asset_return_df[event_date]
            - expected_return_df[event_date]
        )

    cumulative_abnormal_return_df = abnormal_return_df.cumsum()

    residual_std_error_df = event_market_model_results.set_index(
        "event_date"
    ).loc[valid_events, "residual_std_error"]

    relative_day_array = abnormal_return_df.index.values.astype(float)

    car_upper_confidence_bound_df = pd.DataFrame(
        index=abnormal_return_df.index,
        columns=valid_events,
        dtype="float64",
    )

    car_lower_confidence_bound_df = pd.DataFrame(
        index=abnormal_return_df.index,
        columns=valid_events,
        dtype="float64",
    )

    for event_date in valid_events:
        std_error = residual_std_error_df[event_date]
        se_car = std_error * np.sqrt(np.arange(1, len(abnormal_return_df) + 1))

        car_upper_confidence_bound_df[event_date] = (
            cumulative_abnormal_return_df[event_date]
            + z_score * se_car
        )

        car_lower_confidence_bound_df[event_date] = (
            cumulative_abnormal_return_df[event_date]
            - z_score * se_car
        )

    return (
        expected_return_df,
        abnormal_return_df,
        cumulative_abnormal_return_df,
        car_upper_confidence_bound_df,
        car_lower_confidence_bound_df,
    )

In [ ]:
def plot_car_grid(
    car_df,
    car_upper_df,
    car_lower_df,
    save_path=None,
):
    """
    Plot one CAR chart per event with shaded confidence bands using Matplotlib.
    """
    if car_df.empty:
        raise ValueError(
            "car_df is empty. Nothing to plot."
        )

    plt.rcParams["font.family"] = "serif"

    n_events = len(car_df.columns)
    num_rows = 3
    num_cols = int(np.ceil(n_events / num_rows))

    fig, axes = plt.subplots(
        num_rows,
        num_cols,
        figsize=(4 * num_cols, 4 * num_rows),
        squeeze=False,
    )

    axes = axes.ravel()

    for idx, event_date in enumerate(car_df.columns):

        ax = axes[idx]

        ax.plot(
            car_df.index,
            car_df[event_date].values,
            color="blue",
            linewidth=2,
        )

        ax.fill_between(
            car_df.index,
            car_lower_df[event_date].values,
            car_upper_df[event_date].values,
            color="blue",
            alpha=0.3,
            linewidth=0,
        )

        ax.axvline(
            0,
            color="black",
            linewidth=1,
        )

        ax.axhline(
            0,
            color="black",
            linewidth=1,
        )

        ax.set_title(
            pd.Timestamp(event_date).strftime("%Y-%m-%d"),
            fontsize=11,
            fontweight="bold",
            loc="left",
        )

        ax.set_xlabel(
            "Days relative to event",
            fontsize=9,
        )

        ax.set_ylabel(
            "CAR",
            fontsize=9,
        )

    for idx in range(n_events, len(axes)):
        fig.delaxes(axes[idx])

    fig.suptitle(
        "Cumulative Abnormal Returns (CAR) - Scholes-Williams Beta",
        fontsize=14,
        fontweight="bold",
        y=0.995,
    )

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Figure saved to {save_path}")

    return fig

In [ ]:
print("="*80)
print("Estimating Scholes-Williams Market Models for Each Event")
print("="*80)

event_market_model_results, pre_event_abnormal_log_returns = estimate_event_market_models_scholes_williams(
    return_data=eth_crix_log_returns,
    event_dates=EVENT_DATES,
    days_before_start=ESTIMATION_WINDOW_START,
    days_before_end=ESTIMATION_WINDOW_END,
    asset_column=ASSET_TICKER,
    market_column=MARKET_INDEX_TICKER,
)

print("\nScholes-Williams Market Model Results:")
print(event_market_model_results.round(6))

In [ ]:
print("\n" + "="*80)
print("Building Event Windows")
print("="*80)

eth_event_window_log_returns, crix_event_window_log_returns = build_event_log_return_windows(
    return_data=eth_crix_log_returns,
    event_dates=EVENT_DATES,
    limit_before=EVENT_WINDOW_BEFORE,
    limit_after=EVENT_WINDOW_AFTER,
    asset_column=ASSET_TICKER,
    market_column=MARKET_INDEX_TICKER,
)

print(f"\nETH event-window shape: {eth_event_window_log_returns.shape}")
print(f"CRIX event-window shape: {crix_event_window_log_returns.shape}")

print("\nETH Event Window (first few rows):")
eth_event_window_log_returns.head()

In [ ]:
print("\n" + "="*80)
print("Computing Cumulative Abnormal Returns (CAR) with Confidence Intervals")
print("="*80)

(
    eth_expected_log_returns,
    eth_abnormal_log_returns,
    eth_cumulative_abnormal_log_returns,
    eth_car_upper_confidence_bound,
    eth_car_lower_confidence_bound,
) = compute_car_with_ci(
    asset_return_df=eth_event_window_log_returns,
    market_return_df=crix_event_window_log_returns,
    event_market_model_results=event_market_model_results,
    z_score=Z_SCORE,
)

print(f"\nCAR shape: {eth_cumulative_abnormal_log_returns.shape}")
print("\nCAR (first few rows):")
eth_cumulative_abnormal_log_returns.head()

In [ ]:
fig = plot_car_grid(
    car_df=eth_cumulative_abnormal_log_returns,
    car_upper_df=eth_car_upper_confidence_bound,
    car_lower_df=eth_car_lower_confidence_bound,
    save_path="car_grid_scholes_williams.png",
)

plt.show()

In [ ]:
print("\n" + "="*80)
print("Saving Results to CSV Files")
print("="*80)

# Save event market model results
event_market_model_results.to_csv(
    EVENT_MARKET_MODEL_RESULTS_FILE,
    index=False,
)
print(f"\n✓ Saved: {EVENT_MARKET_MODEL_RESULTS_FILE}")

# Save pre-event abnormal returns
pre_event_abnormal_log_returns.to_csv(
    PRE_EVENT_ABNORMAL_LOG_RETURNS_FILE,
    index=True,
)
print(f"✓ Saved: {PRE_EVENT_ABNORMAL_LOG_RETURNS_FILE}")

# Save ETH event window log returns
eth_event_window_log_returns.to_csv(
    ETH_EVENT_WINDOW_LOG_RETURNS_FILE,
    index=True,
)
print(f"✓ Saved: {ETH_EVENT_WINDOW_LOG_RETURNS_FILE}")

# Save CRIX event window log returns
crix_event_window_log_returns.to_csv(
    CRIX_EVENT_WINDOW_LOG_RETURNS_FILE,
    index=True,
)
print(f"✓ Saved: {CRIX_EVENT_WINDOW_LOG_RETURNS_FILE}")

# Save ETH abnormal log returns
eth_abnormal_log_returns.to_csv(
    ETH_ABNORMAL_LOG_RETURNS_FILE,
    index=True,
)
print(f"✓ Saved: {ETH_ABNORMAL_LOG_RETURNS_FILE}")

print("\n" + "="*80)
print("All results saved successfully!")
print("="*80)

print("\nFiles created:")
print(f"  1. {EVENT_MARKET_MODEL_RESULTS_FILE.name}")
print(f"  2. {PRE_EVENT_ABNORMAL_LOG_RETURNS_FILE.name}")
print(f"  3. {ETH_EVENT_WINDOW_LOG_RETURNS_FILE.name}")
print(f"  4. {CRIX_EVENT_WINDOW_LOG_RETURNS_FILE.name}")
print(f"  5. {ETH_ABNORMAL_LOG_RETURNS_FILE.name}")